In [ ]:

import json
from collections import Counter
from datetime import datetime, timedelta
from itertools import product, zip_longest
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import plotly.graph_objects as goa
import regex
import requests
import yaml
from plotly.colors import qualitative, sample_colorscale
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
from datetime import datetime
import pytz
from src.utils import (
    guardarExcel,
    guardarExcelMulti
)

from datetime import timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)
from src.utils import (
    isEmpty,
    loadEstaciones,
    loadLocalizaciones,
    localizeFecha,
    parallelizeFunction,
    rellenarId,
    removeDoubleQuotes,
    splitDataframe,
)
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isEmpty,
    map_cod2name,
    map_name2use_name,
    parallelizeFunction,
    setEF
)
from src.utils.util import (
    loadEstaciones,
    loadEstacionSinCTC
)
from src.processor import (
    XPECProcessor
    
)
from src.utils.topos import getEstacionamientos

In [ ]:
from datetime import timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)
# pd.set_option("future.no_silent_downcasting", True)
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

import argparse
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import regex
import yaml
from tqdm.auto import tqdm

from src.api import cargarHistorico, getHistoricoMOW
from src.processor import LogProcessor
from src.utils import isValidCode, parallelizeFunction, parseDate, rellenarId
from src.api.APIs import (
    getEstadoCirculacionesTecnicas,
    getPlanificacionCirculacionesTecnicas)

In [ ]:
# Tipos de tren que queremos
train_types = {
    "Approach": "APROXIMACIÓN",
    "Arrival": "LLEGADA",
    "Departure": "SALIDA",
    "Elimination": "SUPRESIÓN",
    "End": "FIN",
    "Entry": "ENTRY",
    "Exit": "EXIT",
    "Maneuver": "MANIOBRA",
    "Platform": "ORIGEN",
    "PlatformForecast": "PREVISIÓN",  # "PREDICCIÓN",
    "Stopped": "STOP",
    "TrackingLost": "LOST_TRACK",
}

# Orden lógico de movimientos
mov_sorter = {
    v: k
    for k, v in enumerate(
        [
            "PREVISIÓN",
            "APROXIMACIÓN",
            "EXIT",
            "LLEGADA",
            "FIN",
            "BAJA",
            "ALTA",
            "SALIDA",
            "MANIOBRA_APROXIMACION"
            "MANIOBRA_LLEGADA",
            "MANIOBRA_SALIDA",
        ]
    )
}

In [ ]:
estaciones = loadEstaciones()

In [ ]:
ovi =  estaciones[estaciones["CTC"]  == "OVI"]

In [ ]:
guardarExcel(ovi, "estaciones_ovi.xlsx")

In [ ]:
def cargarHistorico(
    start_date: str,
    end_date: str,
    estaciones: list[str],
    trenes: list[str],
    xSIV: bool = True,
    jCTC: bool = False,
    pro: bool = True,
    maniobra:bool = True
):
    # Comprobamos que la fecha de fin sea después de la de inicio
    if end_date <= start_date:
        end_date = (pd.to_datetime(start_date) + timedelta(days=1)).strftime(
            "%Y-%m-%d %H:%M:%S"
        )

    historico = getHistoricoMOW(
        estaciones=estaciones,
        trenes=trenes,
        inicio=start_date,
        fin=end_date,
        xSIV=xSIV,
        jCTC=jCTC,
        pro=pro,
        maniobra= maniobra
    )
    # historico = historico[
    #     (historico["Fecha"] >= pd.to_datetime(start_date))
    #     & (historico["Fecha"] <= pd.to_datetime(end_date))
    # ]
    # Usamos movimientos auditados
    # historico = historico[
    #     np.invert(historico["FuenteVía"].isin(["PLANNED", "SITRA_PROVIDED"]))
    # ]
    historico = historico[historico["NTécnico"].apply(isValidCode)].dropna(
        subset=["Movimiento"]
    )
    historico["mov_ord"] = historico["Movimiento"].apply(mov_sorter.get)
    return historico


In [ ]:
start_date = "2025-09-21"
end_date = "2025-09-22"
estaciones = ["17000"]

In [ ]:
ntrenes = [rellenarId(el) for el in np.arange(100000)]
historico_pro = cargarHistorico(
    start_date,
    end_date,
    estaciones,
    ntrenes,
    xSIV=True,
    jCTC=False,
    pro=True,
)
historico_pro = historico_pro.sort_values(
    by=["FechaOrigen", "NTécnico", "Fecha", "mov_ord"]
).reset_index(drop=True)

# Añadir información de la fecha
historico_pro["Día"] = historico_pro["Fecha"].dt.date
historico_pro["day_of_week"] = historico_pro["Fecha"].dt.day_of_week
historico_pro["day_of_year"] = historico_pro["Fecha"].dt.day_of_year
historico_pro["week_of_year"] = (historico_pro["day_of_year"] / 7).astype(int)

In [ ]:
estaciones_sin_ctc = loadEstacionSinCTC()

In [ ]:
merge = pd.merge(
    estaciones_sin_ctc, 
    historico_pro, 
    how="left", 
    on="Código"
)

In [ ]:
sin_ctc = merge[~merge["Nombre_x"].isna()]

In [ ]:
sin_ctc.columns

In [ ]:
agrupado = sin_ctc.groupby(["Código", "FuenteMovimiento"]).size().reset_index(name="count").sort_values(by="count", ascending=False)

In [ ]:
agrupado[agrupado["FuenteMovimiento"] == "CTC_MIE"]


In [ ]:
sin_ctc[sin_ctc["Código"] == "03217"]

In [ ]:
agrupado.rename(columns={"Nombre_y":"Nombre"}, inplace=True)

In [ ]:
final = agrupado[agrupado["FuenteMovimiento"].isin(["SITRA","CTC_MIE","MSE"])].copy()

In [ ]:
final[final["FuenteMovimiento"] == "CTC_MIE"]

In [ ]:
final.sort_values(by=["Código"],inplace=True)



In [ ]:
final["FuenteMovimiento"].unique()

In [ ]:
final_1 = pd.merge(
    final,
    estaciones_sin_ctc,
    how="right",
    on=["Código"]
    
)

In [ ]:
final_1.rename(columns={"Nombre_x":"Nombre"}, inplace=True)

In [ ]:
final_1[final_1["FuenteMovimiento"] == "CTC_MIE"]

In [ ]:
final_1 = final_1[["Código","Nombre","FuenteMovimiento","count"]]

In [ ]:
final_1["count"] = final_1["count"].fillna(0).astype(int)

In [ ]:
if {"FuenteMovimiento","Código","Nombre"}.issubset(final_1.columns) and "Código" in estaciones_sin_ctc.columns:
    mapa_nombres = estaciones_sin_ctc.set_index("Código")["Nombre"].astype(str).to_dict()
    mask_nulo = final_1["FuenteMovimiento"].isna()
    if mask_nulo.any():
        final_1.loc[mask_nulo, "Nombre"] = final_1.loc[mask_nulo, "Código"].map(mapa_nombres).fillna(final_1.loc[mask_nulo, "Nombre"])

In [ ]:
final

In [ ]:
# ...existing code...
# crear columnas con el conteo por cada FuenteMovimiento
# usamos 'final' (Código, FuenteMovimiento, count) para pivotar
if final.empty:
    counts = pd.DataFrame(columns=["Código"])  # evitar errores si final está vacío
else:
    counts = (
        final.groupby(["Código", "FuenteMovimiento"])["count"]
        .sum()
        .unstack(fill_value=0)  # una columna por cada FuenteMovimiento
        .reset_index()
    )
counts.columns.name = None

# unir con la lista completa de estaciones (solo Código y Nombre para evitar columnas textuales extras)
final_counts = pd.merge(
    estaciones_sin_ctc[["Código", "Nombre"]],
    counts,
    on="Código",
    how="left",
).fillna(0)

# asegurar tipo entero en los contadores (solo para las columnas que realmente representan conteos)
for c in final_counts.columns:
    if c not in ("Código", "Nombre"):
        # convertir valores numéricos; si hay valores no convertibles, rellenar con 0
        final_counts[c] = pd.to_numeric(final_counts[c], errors="coerce").fillna(0).astype(int)

# actualizar final_1 con la tabla resultado para que se use al guardar
final_2 = final_counts.copy()
# ...existing code...

In [ ]:
final_3 = pd.merge(
    final_2,
    estaciones_sin_ctc,
    on="Código",
    how="left",
)

In [ ]:
final_3

In [ ]:
final_3.drop(columns=["Nombre_y"], inplace=True)

In [ ]:
final_3.rename(columns={"Nombre_x":"Nombre"}, inplace=True)

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe\conteo_sin_ctc\20250928_conteo_sin_ctc.xlsx")
data = {
    "Informe_sin_ctc": final_3
}

In [ ]:
guardarExcelMulti(data, fname) 

In [ ]:
import json
import requests
from src.api.APIs import hacerPeticion
HOSTPATH = "http://info.api.elcano.operaciones.adif/mse-circulations/msecirculations/planning/day/"
data = {"day": pd.to_datetime("2025-09-21").strftime("%Y-%m-%d")}
data = json.dumps(data)

response = hacerPeticion(
    "POST",
    HOSTPATH,
    data=data,
)
res_data = regex.sub(r"\n*data:\s*", ",", response.text)[1:]
res_data = json.loads(f"[{res_data}]")

In [ ]:

def parse_launching_date(ld):
    """
    Convierte launchingDate a pd.Timestamp:
    - Lista/tupla [YYYY, M, D] o [YYYY, M, D, hh, mm, ss]
    - String o cualquier otro formato reconocido por pd.to_datetime
    - Devuelve NaT si no se puede convertir
    """
    try:
        # Caso: lista o tupla
        if isinstance(ld, (list, tuple)) and len(ld) >= 3:
            y, m, d = int(ld[0]), int(ld[1]), int(ld[2])
            if len(ld) >= 6:
                hh, mm, ss = int(ld[3]), int(ld[4]), int(ld[5])
                return pd.Timestamp(year=y, month=m, day=d, hour=hh, minute=mm, second=ss)
            return pd.Timestamp(year=y, month=m, day=d)
        # Caso: otros tipos -> pd.to_datetime intenta convertir
        return pd.to_datetime(ld, errors='coerce')
    except Exception:
        return pd.NaT

In [ ]:
def extract_steps(journey):
    """
    Extrae y transforma los steps de 'journey' a una lista de dicts ya tipados.
    """
    print(journey)
    steps = (journey or {}).get("steps") or []
    out = []
    for s in steps:
        out.append({
            "step": s.get("step"),
            "index": s.get("index"),
            "pointId": s.get("pointId"),
            "parkingTrack": s.get("parkingTrack"),
            "parkingTrackForDeparture": s.get("parkingTrackForDeparture"),
            "stationaryType": s.get("stationaryType"),
            "parity": s.get("parity"),
            "previous": s.get("previous"),
            "next": s.get("next"),
            "technicalStop": s.get("technicalStop"),
            "steps24h": s.get("steps24h"),
            "circulationMode": s.get("circulationMode"),
        })
    return out

In [ ]:

rows = []
for el in res_data:
    cid = el.get("circulationId", {}) or {}
    tecnico = cid.get("number")
    fecha = parse_launching_date(cid.get("launchingDate"))

    steps = (el.get("dayTrain") or {}).get("journey").get("steps") or []
    for s in steps:
        rows.append({
            "NTécnico": tecnico,
            "FechaOrigen": fecha,
            "Secuencia": s.get("step"),
            "Código": s.get("pointId"),   
            "Vía_Planificada": s.get("parkingTrack")
        })

planificacion = pd.DataFrame(rows)

In [ ]:
ruta = Path(r"c:\Users\xiangzhou.zhang\Downloads\Calidad_2 (41).csv")
planificacion = pd.read_csv(ruta)

In [ ]:
planificacion = planificacion[planificacion["Fecha Origen Tren (YYYYMMDD) N"] == "20250921"]

In [ ]:
historico_pro = historico_pro[historico_pro["FechaOrigen"] == "2025-09-21"]

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\ADIF\Elcano - Documentos\Elcano Desarrollo\Info SITRA\planificación_20250922.xlsx")
guardarExcel(planificacion,fname)

In [ ]:
import json
import requests
from src.api.APIs import hacerPeticion
HOSTPATH = "http://info.api.elcano.operaciones.adif/mse-circulations/msecirculations/planning/day/"
data = {"day": pd.to_datetime("2025-09-20").strftime("%Y-%m-%d")}
data = json.dumps(data)

response = hacerPeticion(
    "POST",
    HOSTPATH,
    data=data,
)
res_data = regex.sub(r"\n*data:\s*", ",", response.text)[1:]
res_data = json.loads(f"[{res_data}]")

In [ ]:

rows = []
for el in res_data:
    cid = el.get("circulationId", {}) or {}
    tecnico = cid.get("number")
    fecha = parse_launching_date(cid.get("launchingDate"))

    steps = (el.get("dayTrain") or {}).get("journey").get("steps") or []
    for s in steps:
        rows.append({
            "NTécnico": tecnico,
            "FechaOrigen": fecha,
            "Secuencia": s.get("step"),
            "Código": s.get("pointId"), 
            "Vía_Planificada": s.get("parkingTrack")
        })

planificacion1 = pd.DataFrame(rows)

In [ ]:
import json
import requests
from src.api.APIs import hacerPeticion
HOSTPATH = "http://info.api.elcano.operaciones.adif/mse-circulations/msecirculations/planning/day/"
data = {"day": pd.to_datetime("2025-09-21").strftime("%Y-%m-%d")}
data = json.dumps(data)

response = hacerPeticion(
    "POST",
    HOSTPATH,
    data=data,
)
res_data = regex.sub(r"\n*data:\s*", ",", response.text)[1:]
res_data = json.loads(f"[{res_data}]")

In [ ]:
rows = []
for el in res_data:
    cid = el.get("circulationId", {}) or {}
    tecnico = cid.get("number")
    fecha = parse_launching_date(cid.get("launchingDate"))

    steps = (el.get("dayTrain") or {}).get("journey").get("steps") or []
    for s in steps:
        rows.append({
            "NTécnico": tecnico,
            "FechaOrigen": fecha,
            "Secuencia": s.get("step"),
            "Código": s.get("pointId"),  
            "Vía_Planificada": s.get("parkingTrack")
        })

planificacion2 = pd.DataFrame(rows)

In [ ]:
planificacion = pd.concat([planificacion,planificacion1,planificacion2],ignore_index=True)

In [ ]:
planificacion.sort_values(by=["FechaOrigen"],inplace=True)

In [ ]:
historico_pro

In [ ]:
fname = Path(r"data/Subdirección.csv")

In [ ]:
subdirección = pd.read_csv(fname)

In [ ]:
subdirección


In [ ]:
centro_historico  = pd.merge(
    subdirección,
    historico_pro,
    on = "Código",
    how="right"
)

In [ ]:
historico_mes_planif = pd.merge(
    centro_historico,
    planificacion,
    on = ["FechaOrigen","NTécnico","Código"],
    how="left",
)

In [ ]:
test = historico_pro.copy()

In [ ]:
mov = test['Movimiento'].astype(str).str.strip().str.lower()
has_both = (
    test.assign(_mov=mov)
      .groupby(['FechaOrigen', 'Código', 'NTécnico'])['_mov']
      .transform(lambda s: {'origen', 'salida'}.issubset(set(s)))
)
out_a = test[has_both & mov.isin(['origen', 'salida'])].copy()
origen_test = out_a[out_a["Movimiento"] == "SALIDA"].copy() 
origen_test["Tipo circulación"] = 'Origen'
origen_test.reset_index(drop=True, inplace=True)


In [ ]:
has_both = (
    test.assign(_mov=mov)
      .groupby(['FechaOrigen', 'Código', 'NTécnico'])['_mov']
      .transform(lambda s: {'llegada', 'salida'}.issubset(set(s)))
)
out_a = test[has_both & mov.isin(['llegada', 'salida'])].copy()
paso = out_a[out_a["Movimiento"] == "LLEGADA"].copy() 
paso["Tipo circulación"] = 'Paso'
paso.reset_index(drop=True, inplace=True)


In [ ]:
has_both = (
    test.assign(_mov=mov)
      .groupby(['FechaOrigen', 'Código', 'NTécnico'])['_mov']
      .transform(lambda s: {"llegada",'fin'}.issubset(set(s)))
)
out_a = test[has_both & mov.isin(['llegada', 'fin'])].copy()
fin = out_a[out_a["Movimiento"] == "LLEGADA"].copy() 
fin["Tipo circulación"] = 'Fin'
fin.reset_index(drop=True, inplace=True)

In [ ]:
# origen_1 = origen_test[["CTC","FechaOrigen","LíneaComercial","Código","Nombre","NTécnico","Tipo circulación","Vía","Vía_Planificada"]]
# paso_1 = paso[["CTC","FechaOrigen","LíneaComercial","Código","Nombre","NTécnico","Tipo circulación","Vía","Vía_Planificada"]]
# fin_1 = fin[["CTC","FechaOrigen","LíneaComercial","Código","Nombre","NTécnico","Tipo circulación","Vía","Vía_Planificada"]]
origen_1 = origen_test[["CTC","FechaOrigen","LíneaComercial","Código","Nombre","NTécnico","Tipo circulación","Vía"]]
paso_1 = paso[["CTC","FechaOrigen","LíneaComercial","Código","Nombre","NTécnico","Tipo circulación","Vía"]]
fin_1 = fin[["CTC","FechaOrigen","LíneaComercial","Código","Nombre","NTécnico","Tipo circulación","Vía"]]

In [ ]:
paso[origen_1["Vía"] == "9B"]

In [ ]:
estaciones_sin_ctc = loadEstaciones()

In [ ]:
estaciones

In [ ]:
circulación_mes = pd.concat([origen_1,fin_1,paso_1],ignore_index = True)

# circulación_mes["Vía_Planificada"] = circulación_mes["Vía_Planificada"].apply(
#     lambda x: str(x) if pd.notnull(x) else x
# )
# circulación_mes["Vía"] = circulación_mes["Vía"].apply(
#     lambda x: str(x) if pd.notnull(x) else x

# )


In [ ]:
circulación_mes['Era la que tenía planificada?'] = circulación_mes.apply(
    lambda row: 'N/A' if pd.isna(row['Vía_Planificada']) else row['Vía'] == row['Vía_Planificada'],
    axis=1
)


In [ ]:
test2= historico_pro.sort_values(by=["NTécnico","Fecha"]).copy()

In [ ]:
sub_dfs = [group for _, group in test2.groupby('NTécnico')]
dfs = []
for df in sub_dfs:
    Fecha1 = [group for _, group in df.groupby('FechaOrigen')]
    dfs.append(Fecha1)

In [ ]:
from itertools import chain

if hasattr(dfs, "flatten"):
    dddf = dfs.flatten()  # por si dfs fuera un ndarray
else:
    dddf = list(chain.from_iterable(dfs))

In [ ]:
ddfs = []
for df in dddf:
    codigo = [group for _, group in df.groupby('Código')]
    ddfs.append(codigo)

In [ ]:
ddfs[3][0]

In [ ]:
información_adicional = []
for tren in ddfs:
    for elemento in tren:
        if isinstance(elemento, pd.DataFrame) and 'Movimiento' in elemento.columns:
            mask_aproximacion = elemento['Movimiento'].str.contains(
                r'APROXIMACIÓN|PREVISIÓN', 
                case=False, 
                na=False,
                regex=True
            )
            mask_llegada = elemento['Movimiento'].str.contains(
                r'LLEGADA', 
                case=False, 
                na=False,
                regex=True
            )
            mask_origen = elemento['Movimiento'].str.contains(
                r'ORIGEN', 
                case=False, 
                na=False,
                regex=True
            )
            mask_salida= elemento['Movimiento'].str.contains(
                r'SALIDA', 
                case=False, 
                na=False,
                regex=True
            )

            aproximacion = elemento[mask_aproximacion]
            llegada = elemento[mask_llegada]
            origen = elemento[mask_origen]
            salida = elemento[mask_salida]
            # print("aproximacion:",aproximacion)
            # print("llegada:",llegada)
            # print("origen:",origen)
            # print("salida:",salida)
            
            
            # Verificar primero si hay registros de origen
            if not origen.empty:
                print("origen")
                if origen["FuenteVía"].iloc[0] == "CTC":
                    print("CTC")
                    tiempo_prevision = origen["Fecha"].iloc[0]
                    NTécnico = origen["NTécnico"].iloc[0]
                    Código = origen["Código"].iloc[0]
                    print("NTécnico;",NTécnico)
                    Fecha = origen["FechaOrigen"].iloc[0]
                    # Buscar la llegada correspondiente
                    salida_correspondiente = salida
                    if not salida.empty:
                        tiempo_salida = salida["Fecha"].iloc[0]
                        tiempo_anticipación_CTC = tiempo_salida - tiempo_prevision
                    else:
                        tiempo_anticipación_CTC = "NA"
                    Sitra= False
                    Anticipación_sitra = "NA"
                    CTC = True

                    column = {"NTécnico":NTécnico,"FechaOrigen":Fecha,"Código":Código,"Se ha anticipado por CTC":CTC,"Tiempo de anticipación CTC":tiempo_anticipación_CTC,"Se ha anticipado por Sitra":Sitra,"Tiempo de anticipación SITRA":Anticipación_sitra}
                    información_adicional.append(column)
                elif origen["FuenteVía"].iloc[0] in ["SITRA_AUDITED", "SITRA_PROVIDED"]:
                        print("Sitra")
                        tiempo_prevision = origen["Fecha"].iloc[0]
                        NTécnico = origen["NTécnico"].iloc[0]
                        Fecha = origen["FechaOrigen"].iloc[0]
                        Código = origen["Código"].iloc[0]
                        print("NTécnico;",NTécnico)
                        salida_correspondiente = salida
                        print("tiempo_Previson:",tiempo_prevision)
                        Sitra= True
                        if not salida.empty:
                            tiempo_salida = salida["Fecha"].iloc[0]
                            Anticipación_sitra = tiempo_salida - tiempo_prevision
                        else:
                            Anticipación_sitra = "NA"
                        CTC = False
                        tiempo_anticipación_CTC ="NA"
                        column = {"NTécnico":NTécnico,"FechaOrigen":Fecha,"Código":Código,"Se ha anticipado por CTC":CTC,"Tiempo de anticipación CTC":tiempo_anticipación_CTC,"Se ha anticipado por Sitra":Sitra,"Tiempo de anticipación SITRA":Anticipación_sitra}
                        información_adicional.append(column)
            
            # Si no hay origen pero hay aproximación
            elif not aproximacion.empty:
                print("aproximación")
                if aproximacion["FuenteVía"].iloc[0] == "CTC":
                    print("CTC")
                    tiempo_prevision = aproximacion["Fecha"].iloc[0]
                    NTécnico = aproximacion["NTécnico"].iloc[0]
                    print("NTécnico;",NTécnico)
                    Fecha = aproximacion["FechaOrigen"].iloc[0]
                    Código = aproximacion["Código"].iloc[0]
                    # Buscar la llegada correspondiente
                    llegada_correspondiente = llegada
                    # print("llegada_correspondiente",llegada_correspondiente)
                    # print(llegada_correspondiente)
                    # print("tiempo_llegada:",tiempo_llegada)
                    print("tiempo_prevision",tiempo_prevision)
                    if not llegada.empty:
                        tiempo_llegada = llegada["Fecha"].iloc[0]
                        tiempo_anticipación_CTC = tiempo_llegada - tiempo_prevision
                    else:
                        tiempo_anticipación_CTC = "NA"
                    Sitra= False
                    Anticipación_sitra = "NA"
                    CTC = True

                    column = {"NTécnico":NTécnico,"FechaOrigen":Fecha,"Código":Código,"Se ha anticipado por CTC":CTC,"Tiempo de anticipación CTC":tiempo_anticipación_CTC,"Se ha anticipado por Sitra":Sitra,"Tiempo de anticipación SITRA":Anticipación_sitra}
                    información_adicional.append(column)
                        
                
                elif aproximacion["FuenteVía"].iloc[0] in ["SITRA_AUDITED", "SITRA_PROVIDED"]:
                        print("Sitra")
                        tiempo_prevision = aproximacion["Fecha"].iloc[0]
                        NTécnico = aproximacion["NTécnico"].iloc[0]
                        print("NTécnico;",NTécnico)
                        Fecha = aproximacion["FechaOrigen"].iloc[0]
                        Código = aproximacion["Código"].iloc[0]
                        # Buscar la llegada correspondiente
                        llegada_correspondiente = llegada
                        Sitra= True
                        if not llegada.empty:
                            tiempo_llegada = llegada["Fecha"].iloc[0]
                            Anticipación_sitra = tiempo_llegada - tiempo_prevision
                        else:
                            Anticipación_sitra = "NA"
                        CTC = False
                        tiempo_anticipación_CTC = "NA"
                        column = {"NTécnico":NTécnico,"FechaOrigen":Fecha,"Código":Código,"Se ha anticipado por CTC":CTC,"Tiempo de anticipación CTC":tiempo_anticipación_CTC,"Se ha anticipado por Sitra":Sitra,"Tiempo de anticipación SITRA":Anticipación_sitra}
                        información_adicional.append(column) 

                                        
        else:
            print("El elemento no es un DataFrame o no tiene columna 'Movimiento'")


In [ ]:
df= pd.DataFrame(información_adicional)

In [ ]:
df

In [ ]:
test = pd.merge(
    circulación_mes,
    df,
    on=["NTécnico","FechaOrigen","Código"],
    how = "left"
)

In [ ]:
test.rename(columns={"Vía":"Vía Estacionmamiento real","Via Estacionamiento":"Vía Estacionamiento planif"}, inplace=True)

In [ ]:
test

In [ ]:
test["FechaOrigen"] = test["FechaOrigen"].astype(str)

In [ ]:
def convertir_a_mm_ss(valor):
    if pd.isna(valor) or valor == "NA":
        return valor
    try:
        # Extraer la parte del tiempo (después de 'days ')
        tiempo = valor.split()[-1]
        horas, minutos, segundos = map(int, tiempo.split(':'))
        # Convertir todo a segundos y luego a minutos:segundos
        total_segundos = horas * 3600 + minutos * 60 + segundos
        mm = total_segundos // 60
        ss = total_segundos % 60
        return f"{mm:02d}:{ss:02d}"
    except:
        return valor  # En caso de que el formato no sea el esperado


In [ ]:
test["Tiempo de anticipación CTC"]= test['Tiempo de anticipación CTC'].apply(convertir_a_mm_ss)


In [ ]:
test["Tiempo de anticipación SITRA"]= test['Tiempo de anticipación SITRA'].apply(convertir_a_mm_ss)

In [ ]:
test

In [ ]:
test["Tiempo de anticipación CTC"] = test["Tiempo de anticipación CTC"].astype(str)
test["Tiempo de anticipación SITRA"] = test["Tiempo de anticipación SITRA"].astype(str)

In [ ]:
# test.rename(columns={"Tiempo de anticipación CTC":"Tiempo de anticipación CTC(mm:ss)","Tiempo de anticipación SITRA":"Tiempo de anticipación SITRA(mm:ss)"}, inplace=True)

<h1>Resumen diario</h1>


In [ ]:
# diario = test.copy()

In [ ]:
# conteo = diario.groupby(["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?"]).size().reset_index(name='count')

In [ ]:
# estaciones = loadEstaciones()

In [ ]:
# estaciones = estaciones[["CTC","Código"]].copy()

In [ ]:
# merge = pd.merge(
#     estaciones,
#     conteo,
#     on=["Código"],
#     how = "right"
# )

In [ ]:
# diario.head(4)

In [ ]:
# conteo_1 = diario.groupby(["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?"]).size().reset_index(name='count')

In [ ]:
# coincide = conteo_1[conteo_1["Era la que tenía planificada?"] == True]

In [ ]:
# NoConcide = conteo_1[conteo_1["Era la que tenía planificada?"] == False]

In [ ]:
# merge.rename(columns={"count":"Número total de circulaciones que han llegado realmente a esa vía "}, inplace=True)

In [ ]:
# merge_1 = pd.merge(
#     merge,
#     NoConcide,
#     on=["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?"],
#     how = "left"
# )

In [ ]:
# merge_1 = pd.merge(
#     merge_1,
#     coincide,
#     on=["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?"],
#     how = "left"
# )

In [ ]:
# merge_1.rename(columns={"count_x":"Número de circulaciones que han llegado a esa vía y no la tenían planificada","count_y":"Número de circulaciones que han llegado a esa vía y si la tenían planificada"}, inplace=True)

In [ ]:
# merge_1["Número de circulaciones que han llegado a esa vía y no la tenían planificada"] = merge_1.apply(
#     lambda row: 0 if  pd.isna(row["Número de circulaciones que han llegado a esa vía y no la tenían planificada"])else row["Número de circulaciones que han llegado a esa vía y no la tenían planificada"], axis=1)


In [ ]:
# merge_1["Número de circulaciones que han llegado a esa vía y si la tenían planificada"] = merge_1.apply(lambda row: 0 if  pd.isna(
#     row["Número de circulaciones que han llegado a esa vía y si la tenían planificada"])else row["Número de circulaciones que han llegado a esa vía y si la tenían planificada"], axis=1)


In [ ]:
# sitra = diario.groupby(["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?","Se ha anticipado por Sitra"]).size().reset_index(name='count')

In [ ]:
# anticipación_sitra = sitra[sitra["Se ha anticipado por Sitra"]].copy()

In [ ]:
# merge_1 = pd.merge(
#     merge_1,
#     anticipación_sitra,
#     on=["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?"],
#     how = "left"
# )

In [ ]:
# merge_1.drop(columns=["Se ha anticipado por Sitra"], inplace=True)

In [ ]:
# merge_1.rename(columns={"count":"Número de circulaciones con Anticipación SITRA"}, inplace=True)

In [ ]:
# merge_1["Número de circulaciones con Anticipación SITRA"] = merge_1["Número de circulaciones con Anticipación SITRA"].apply(
#     lambda x: 0 if pd.isna(x) else x
# )

In [ ]:
# merge_1["Número de circulaciones que han llegado a esa vía y si la tenían planificada"] = merge_1["Número de circulaciones que han llegado a esa vía y si la tenían planificada"].apply(
#     lambda x: 0 if pd.isna(x) else x)


In [ ]:
# tiempo_sitra  = diario.groupby(["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Tiempo de anticipación SITRA","Era la que tenía planificada?"]).size().reset_index(name='count')

In [ ]:
# anticipación_sitra = tiempo_sitra[~tiempo_sitra["Tiempo de anticipación SITRA"].isin(["NA","nan"])].copy()

In [ ]:
# anticipación_sitra.drop(columns=["count"], inplace=True)

In [ ]:
# sub_dfs = [group for _, group in anticipación_sitra.groupby(["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real"])]


In [ ]:
# for df in sub_dfs:
#     df['Tiempo de anticipación SITRA'] = pd.to_timedelta(df['Tiempo de anticipación SITRA'])
#     # print(df['Tiempo de anticipación SITRA'])
#     df['Tiempo de antelación de anuncio SITRA'] = df['Tiempo de anticipación SITRA'].mean()
#     df['Tiempo de antelación de anuncio SITRA'] = df['Tiempo de antelación de anuncio SITRA'].apply(lambda x: f"{int(x.total_seconds() // 60):02}:{int(x.total_seconds() % 60):02}")


In [ ]:
# df_completo = pd.concat(sub_dfs, ignore_index=True)

In [ ]:
# df_completo.drop(columns=["Tiempo de anticipación SITRA"], inplace=True)

In [ ]:
# df_completo

In [ ]:
# merge_1 = pd.merge(
#     merge_1,
#     df_completo,
#     on=["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?"],
#     how = "left"
# )

In [ ]:
# merge_1[~merge_1["Tiempo de antelación de anuncio SITRA"].isna()]

In [ ]:
# merge_1["Tiempo de antelación de anuncio SITRA"] = merge_1["Tiempo de antelación de anuncio SITRA"].apply(
#     lambda x: 0 if pd.isna(x) else x)

In [ ]:
# merge_1[~merge_1["Tiempo de antelación de anuncio SITRA"].isin([0])]

In [ ]:
# ctc = diario.groupby(["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?","Se ha anticipado por CTC"]).size().reset_index(name='Número total de circulaciones con estimación de vía auditada CTC')

In [ ]:
# anticipación_ctc= ctc[ctc["Se ha anticipado por CTC"]].copy()

In [ ]:
# anticipación_ctc

In [ ]:
# merge_1 = pd.merge( 
#     merge_1,
#     anticipación_ctc,
#     on=["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?"],
#     how = "left"
# )

In [ ]:
# merge_1.drop(columns=["Se ha anticipado por CTC"], inplace=True)

In [ ]:
# tiempo_ctc  = diario.groupby(["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Tiempo de anticipación CTC","Era la que tenía planificada?"]).size().reset_index(name='count')

In [ ]:
# anticipación_ctc = tiempo_ctc[~tiempo_ctc["Tiempo de anticipación CTC"].isin(["NA","nan"])].copy()

In [ ]:
# anticipación_ctc.drop(columns=["count"], inplace=True)

In [ ]:
# anticipación_ctc

In [ ]:
# sub_dfs = [group for _, group in anticipación_ctc.groupby(["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?"])]


In [ ]:
# for df in sub_dfs:
#     df['Tiempo de anticipación CTC'] = pd.to_timedelta(df['Tiempo de anticipación CTC'])
#     # print(df['Tiempo de anticipación SITRA'])
#     df['Tiempo de antelación de anuncio CTC'] = df['Tiempo de anticipación CTC'].mean()
#     df['Tiempo de antelación de anuncio CTC'] = df['Tiempo de antelación de anuncio CTC'].apply(lambda x: f"{int(x.total_seconds() // 60):02}:{int(x.total_seconds() % 60):02}")

In [ ]:
# df_completo = pd.concat(sub_dfs, ignore_index=True) 

In [ ]:
# df_completo.drop(columns=["Tiempo de anticipación CTC"], inplace=True)

In [ ]:
# df_completo.drop_duplicates(subset=["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?"], inplace=True)

In [ ]:
# merge_1 = pd.merge(
#     merge_1,
#     df_completo,
#     on=["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?"],
#     how = "left"
# )

In [ ]:
data={"detalle_tren":test,
    #   "resumen_diario":merge_1,
}

In [ ]:
fname =Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\Chamartin.xlsx")

In [ ]:
guardarExcelMulti(data,fname)

In [ ]:
# circulación = test.groupby(["FechaOrigen","Tipo circulación","Código"]).size().reset_index(name='Número total de circulación')
circulación = test.groupby(["FechaOrigen","Tipo circulación","Código","Vía Estacionmamiento real"]).size().reset_index(name='Número total de circulación')

In [ ]:
circulación_origen = circulación[circulación["Tipo circulación"] == "Origen"].copy()

In [ ]:
anticipación_origen = test[test["Tipo circulación"] == "Origen"].copy()


In [ ]:
anticipación_origen = anticipación_origen[anticipación_origen["Se ha anticipado por CTC"] == True].copy()

In [ ]:
anticipación_origen['Tiempo de anticipación CTC'] = pd.to_timedelta(anticipación_origen['Tiempo de anticipación CTC'])

In [ ]:
# anticipación_origen =anticipación_origen[anticipación_origen["Tiempo de anticipación CTC"] < timedelta(minutes=25)]

In [ ]:
anticipación_origen.sort_values(by=["FechaOrigen"],inplace=True) 

In [ ]:
#sub_dfs = [group for _, group in anticipación_origen.groupby(["FechaOrigen","Código"])]
sub_dfs = [group for _, group in anticipación_origen.groupby(["FechaOrigen","Código","Vía Estacionmamiento real"])]


In [ ]:
tiempo_medio = []
for df in sub_dfs:
    df['Tiempo de anticipación CTC'] = pd.to_timedelta(df['Tiempo de anticipación CTC'])
    fecha = df['FechaOrigen'].iloc[0]
    tiempo_promedio = df['Tiempo de anticipación CTC'].mean()
    tiempo_promedio = f"{int(tiempo_promedio.total_seconds() // 60):02}:{int(tiempo_promedio.total_seconds() % 60):02}"

    tiempo_medio.append({
        'FechaOrigen': fecha,
        'Vía Estacionmamiento real' : df['Vía Estacionmamiento real'].iloc[0],
        "Código": df['Código'].iloc[0],
        'Tiempo medio anticipación CTC': tiempo_promedio
    })


In [ ]:
tiempo_origen = pd.DataFrame(tiempo_medio)


In [ ]:
tiempo_origen

In [ ]:
circulación_origen.sort_values(by=["FechaOrigen"], inplace=True)

In [ ]:
fiabilidad_origen = test[test["Tipo circulación"] == "Origen"].copy()

In [ ]:
fiabilidad_origen_1 = fiabilidad_origen.groupby(["FechaOrigen","Era la que tenía planificada?","Código"]).size().reset_index(name='count')

In [ ]:
sub_dfs = [group for _, group in fiabilidad_origen_1.groupby(["FechaOrigen","Código"])]

In [ ]:
fiabilidades = []
for df in sub_dfs:
    df['count'] = df['count'].astype(int)
    fecha = df['FechaOrigen'].iloc[0]
    total = df['count'].sum()
    if total > 0:
        porcentaje_planificado = (df[df['Era la que tenía planificada?'] == True]['count'].sum() / total) * 100
        porcentaje_no_planificado = (df[df['Era la que tenía planificada?'] == False]['count'].sum() / total) * 100
    else:
        porcentaje_planificado = 0
        porcentaje_no_planificado = 0
    porcentaje_planificado = round(porcentaje_planificado, 2)
    porcentaje_no_planificado = round(porcentaje_no_planificado, 2)

    fiabilidades.append({
        'FechaOrigen': fecha,
        "Código": df['Código'].iloc[0],
        'Fiabilidad vía planificada(%)': porcentaje_planificado,

    })
    


In [ ]:
porcentaje_fiabilidad = pd.DataFrame(fiabilidades)

In [ ]:
circulación_origen["FechaOrigen"] = circulación_origen["FechaOrigen"].astype(str)
tiempo_origen["FechaOrigen"] = tiempo_origen["FechaOrigen"].astype(str)
estadistica = pd.merge(
    circulación_origen,
    tiempo_origen,
    on=["FechaOrigen","Código","Vía Estacionmamiento real"],
    how = "left"
)

In [ ]:
estadistica

In [ ]:
porcentaje_fiabilidad["FechaOrigen"]= porcentaje_fiabilidad["FechaOrigen"].astype(str)
estadistica = pd.merge(
    estadistica,
    porcentaje_fiabilidad,
    on=["FechaOrigen","Código"],
    how="left"
)

In [ ]:
circulación_destino = circulación[circulación["Tipo circulación"] == "Fin"].copy()
anticipación_destino = test[test["Tipo circulación"] == "Fin"].copy()
anticipación_destino = anticipación_destino[anticipación_destino["Se ha anticipado por CTC"] == True].copy()
anticipación_destino['Tiempo de anticipación CTC'] = pd.to_timedelta(anticipación_destino['Tiempo de anticipación CTC'])
sub_dfs = [group for _, group in anticipación_destino.groupby(["FechaOrigen","Código"])]

In [ ]:
tiempo_medio = []
for df in sub_dfs:
    df['Tiempo de anticipación CTC'] = pd.to_timedelta(df['Tiempo de anticipación CTC'])
    fecha = df['FechaOrigen'].iloc[0]
    tiempo_promedio = df['Tiempo de anticipación CTC'].mean()
    tiempo_promedio = f"{int(tiempo_promedio.total_seconds() // 60):02}:{int(tiempo_promedio.total_seconds() % 60):02}"

    tiempo_medio.append({
        'FechaOrigen': fecha,
        "Código": df['Código'].iloc[0],
        'Tiempo medio anticipación CTC': tiempo_promedio
    })


In [ ]:
tiempo_destino = pd.DataFrame(tiempo_medio)

In [ ]:
circulación_destino.sort_values(by=["FechaOrigen"], inplace=True)

In [ ]:
fiabilidad_destino = test[test["Tipo circulación"] == "Fin"].copy()
fiabilidad_destino_1 = fiabilidad_destino.groupby(["FechaOrigen","Era la que tenía planificada?","Código"]).size().reset_index(name='count')

In [ ]:
sub_dfs = [group for _, group in fiabilidad_destino_1.groupby(["FechaOrigen","Código"])]

In [ ]:
fiabilidades = []
for df in sub_dfs:
    df['count'] = df['count'].astype(int)
    fecha = df['FechaOrigen'].iloc[0]
    total = df['count'].sum()
    if total > 0:
        porcentaje_planificado = (df[df['Era la que tenía planificada?'] == True]['count'].sum() / total) * 100
        porcentaje_no_planificado = (df[df['Era la que tenía planificada?'] == False]['count'].sum() / total) * 100
    else:
        porcentaje_planificado = 0
        porcentaje_no_planificado = 0
    porcentaje_planificado = round(porcentaje_planificado, 2)
    porcentaje_no_planificado = round(porcentaje_no_planificado, 2)

    fiabilidades.append({
        'FechaOrigen': fecha,
        "Código": df['Código'].iloc[0],
        'Fiabilidad vía planificada(%)': porcentaje_planificado,
    })
porcentaje_fiabilidad = pd.DataFrame(fiabilidades)

In [ ]:
tiempo_destino

In [ ]:
circulación_destino["FechaOrigen"] = circulación_destino["FechaOrigen"].astype(str)
tiempo_destino["FechaOrigen"] = tiempo_destino["FechaOrigen"].astype(str)
estadistica_destino = pd.merge(
    circulación_destino,
    tiempo_destino,
    on=["FechaOrigen","Código"],
    how = "left"
)

In [ ]:
estadistica_destino = pd.merge(
    estadistica_destino,
    porcentaje_fiabilidad,
    on=["FechaOrigen","Código"],
    how="left"
)

In [ ]:
circulación_Paso = circulación[circulación["Tipo circulación"] == "Paso"].copy()
anticipación_Paso = test[test["Tipo circulación"] == "Paso"].copy()
anticipación_Paso = anticipación_Paso[anticipación_Paso["Se ha anticipado por CTC"] == True].copy()
anticipación_Paso['Tiempo de anticipación CTC'] = pd.to_timedelta(anticipación_Paso['Tiempo de anticipación CTC'])
sub_dfs = [group for _, group in anticipación_Paso.groupby(["FechaOrigen","Código",'Vía Estacionmamiento real'])]

In [ ]:
tiempo_medio = []
for df in sub_dfs:
    df['Tiempo de anticipación CTC'] = pd.to_timedelta(df['Tiempo de anticipación CTC'])
    fecha = df['FechaOrigen'].iloc[0]
    tiempo_promedio = df['Tiempo de anticipación CTC'].mean()
    tiempo_promedio = f"{int(tiempo_promedio.total_seconds() // 60):02}:{int(tiempo_promedio.total_seconds() % 60):02}"

    tiempo_medio.append({
        'FechaOrigen': fecha,
        "Código": df['Código'].iloc[0],
        'Vía Estacionmamiento real': df["Vía Estacionmamiento real"].iloc[0],
        'Tiempo medio anticipación CTC': tiempo_promedio
    })
tiempo_paso = pd.DataFrame(tiempo_medio)

In [ ]:
circulación_Paso.sort_values(by=["FechaOrigen"], inplace=True)

In [ ]:
fiabilidad_paso = test[test["Tipo circulación"] == "Paso"].copy()
fiabilidad_paso_1 = fiabilidad_paso.groupby(["FechaOrigen","Código","Era la que tenía planificada?"]).size().reset_index(name='count')

In [ ]:
sub_dfs = [group for _, group in fiabilidad_paso_1.groupby(["FechaOrigen","Código"])]

In [ ]:
fiabilidades = []
for df in sub_dfs:
    df['count'] = df['count'].astype(int)
    fecha = df['FechaOrigen'].iloc[0]
    total = df['count'].sum()
    if total > 0:
        porcentaje_planificado = (df[df['Era la que tenía planificada?'] == True]['count'].sum() / total) * 100
        porcentaje_no_planificado = (df[df['Era la que tenía planificada?'] == False]['count'].sum() / total) * 100
    else:
        porcentaje_planificado = 0
        porcentaje_no_planificado = 0
    porcentaje_planificado = round(porcentaje_planificado, 2)
    porcentaje_no_planificado = round(porcentaje_no_planificado, 2)

    fiabilidades.append({
        'FechaOrigen': fecha,
        'Código': df['Código'].iloc[0],
        'Fiabilidad vía planificada(%)': porcentaje_planificado,
    })
porcentaje_fiabilidad = pd.DataFrame(fiabilidades)

In [ ]:
estadistica_paso = pd.merge(
    circulación_Paso,
    tiempo_paso,
    on=["FechaOrigen","Código","Vía Estacionmamiento real"],
    how = "left"
)

In [ ]:
estadistica_paso = pd.merge(
    estadistica_paso,
    porcentaje_fiabilidad,
    on=["FechaOrigen","Código"],
    how="left"
)

In [ ]:
estadistica_total = pd.concat(
    # [estadistica, estadistica_destino,estadistica_paso], ignore_index=True
    [estadistica,estadistica_paso], ignore_index=True
)

In [ ]:
# estadistica_total.drop(columns=["Nombre_y"],inplace=True)
# estadistica_total.rename(columns={"Nombre_x":"Nombre"},inplace=True)

In [ ]:
estadistica_total

In [ ]:
fname = Path(r"data/Subdirección.csv")
subdirección = pd.read_csv(fname)

In [ ]:
estaciones= loadEstaciones()

In [ ]:
estaciones2 = loadEstacionSinCTC()

In [ ]:
estaciones2 =estaciones2[["Código","Nombre"]].copy()

In [ ]:
estaciones = estaciones[["Código","Nombre"]].copy()

In [ ]:
estaciones = pd.concat([estaciones,estaciones2],ignore_index=True)

In [ ]:
estaciones[estaciones["Código"] == "70805"]

In [ ]:
estaciones = historico_pro[["Código","Nombre"]].copy()

In [ ]:
estaciones.drop_duplicates(subset=["Código","Nombre"], inplace=True)

In [ ]:
info = pd.merge(
    subdirección,
    estaciones,
    on =["Código"],
    how = "left"
)

In [ ]:
info[info["Código"] == "70113"]

In [ ]:
info = info.drop_duplicates(subset=["Código", "Nombre"], keep="first")

In [ ]:
estadistica = pd.merge(
    estadistica_total,
    info,
    on =["Código"],
    how= "left",
)

In [ ]:
estadistica = estadistica[["FechaOrigen","Subdirección","Nombre","Tipo circulación","Código","Número total de circulación","Tiempo medio anticipación CTC","Fiabilidad vía planificada(%)"]]

In [ ]:
estadistica

In [ ]:
data = {
    "estadistica": estadistica_total,
    "detalle_tren_diario":test,
    # "resumen_diario":merge_1,
    
}
# fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe\Anticipación_CTC\202509_Anticipación_ctc.xlsx")
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe\Anticipación_CTC\chamartin_nueva.xlsx")

In [ ]:
guardarExcelMulti(data, fname)

In [ ]:
estadistica[estadistica["Código"] == "71902"]

In [ ]:
historico_pro[historico_pro["Código"] == "71902"]